In [ ]:
!pip install pygame
import pygame, csv, time, random, os, json

# --- CONFIGURATION ---
SCREEN_WIDTH = 1400
SCREEN_HEIGHT = 800
CURSOR_RADIUS = 22
TARGET_RADIUS = 14
FPS = 60
FILE_NAME = "experiment_data.csv"
RECORDING_DIR = "recordings"
NUM_ITERATIONS = 20

# Colors
WHITE = (255,255,255)
BLACK = (0,0,0)
RED = (255,0,0)
BLUE = (0,0,255)
GRAY = (128,128,128)

# Game Modes
INDIVIDUAL = "individual"
COOPERATIVE = "cooperative"
PLAYBACK = "playback"
AI = "ai"

# AI Control modes (which axis the AI controls)
AI_CONTROLS_VERTICAL = 1
AI_CONTROLS_HORIZONTAL = 0

class ExperimentGame:

    def __init__(self, mode=INDIVIDUAL, recording_file=None, iteration=1, ai_axis=None):

        pygame.init()
        self.screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        pygame.display.set_caption("CogSci Joint Action Task")
        self.clock = pygame.time.Clock()

        self.running = True
        self.mode = mode
        self.iteration = iteration
        self.target_hit = False

        # Positions
        self.cursor_pos = [SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2]

        margin = 100
        self.target_pos = [
            random.randint(margin, SCREEN_WIDTH - margin),
            random.randint(margin, SCREEN_HEIGHT - margin)
        ]

        # Velocity summing variables
        self.human_v = [0,0]
        self.partner_v = [0,0]

        # Data Logging Setup
        self.data_log = []
        self.start_time = time.time()

        self.recording_file = recording_file

        # Playback Setup
        self.playback_data = []
        self.playback_index = 0

        if mode == PLAYBACK and recording_file:
            self.load_recording(recording_file)

        # AI Setup
        self.ai_control_axis = ai_axis if ai_axis is not None else random.randint(0,1)

    def load_recording(self, filename):

        """Load pre-recorded movements from a JSON file"""

        try:

            with open(filename, 'r') as f:
                self.playback_data = json.load(f)

            print(f"Loaded recording with {len(self.playback_data)} frames")

        except FileNotFoundError:

            print(f"Recording file {filename} not found.")
            self.playback_data = []

    def get_ai_input(self):

        """AI agent that controls either horizontal or vertical movement"""

        dx = self.target_pos[0] - self.cursor_pos[0]
        dy = self.target_pos[1] - self.cursor_pos[1]

        # AI strength (K)
        k = 0.12

        if self.ai_control_axis == AI_CONTROLS_VERTICAL:

            # AI controls up/down only
            return [0, dy * k]

        else:

            # AI controls left/right only
            return [dx * k, 0]

    def get_playback_input(self):

        """Get pre-recorded movement for current frame"""

        if self.playback_index < len(self.playback_data):

            frame = self.playback_data[self.playback_index]
            self.playback_index += 1

            return [frame['p_vx'], frame['p_vy']]

        return [0,0]

    def log_frame(self):

        """Log frame data"""

        elapsed = time.time() - self.start_time

        self.data_log.append([
            elapsed,
            self.cursor_pos[0],
            self.cursor_pos[1],
            self.human_v[0],
            self.human_v[1],
            self.partner_v[0],
            self.partner_v[1],
            self.target_pos[0],
            self.target_pos[1]
        ])

    def save_data(self):

        """Save experiment data to CSV"""

        keys = ["timestamp","cursor_x","cursor_y","h_vx","h_vy","p_vx","p_vy","target_x","target_y"]

        if not os.path.exists(FILE_NAME):

            with open(FILE_NAME, "w", newline="") as f:

                writer = csv.writer(f)
                writer.writerow(keys)

        with open(FILE_NAME, "a", newline="") as f:

            writer = csv.writer(f)
            writer.writerows(self.data_log)

        print(f"Data saved to {FILE_NAME}")

    def draw_text(self, text, font, color, surface, x, y):

        """Draw text on the screen"""

        textobj = font.render(text, True, color)
        textrect = textobj.get_rect()
        textrect.topleft = (x, y)
        surface.blit(textobj, textrect)

    def run(self):

        """Run one iteration - ends when target is hit"""

        font = pygame.font.Font(None, 24)
        speed = 20

        while self.running and not self.target_hit:

            self.screen.fill(WHITE)

            # Reset movement every frame
            self.human_v = [0,0]
            self.partner_v = [0,0]

            # 1. Event Handling
            for event in pygame.event.get():

                if event.type == pygame.QUIT:
                    self.running = False

                if event.type == pygame.KEYDOWN:

                    if event.key == pygame.K_ESCAPE:
                        self.running = False

                    # INDIVIDUAL MODE
                    if self.mode == INDIVIDUAL:

                        # Individual mode: player controls all axes
                        if event.key == pygame.K_LEFT:
                            self.human_v[0] = -speed

                        elif event.key == pygame.K_RIGHT:
                            self.human_v[0] = speed

                        elif event.key == pygame.K_UP:
                            self.human_v[1] = -speed

                        elif event.key == pygame.K_DOWN:
                            self.human_v[1] = speed

                    # COOPERATIVE MODE
                    elif self.mode == COOPERATIVE:

                        # Player 1 controls left/right
                        if event.key == pygame.K_LEFT:
                            self.human_v[0] = -speed

                        elif event.key == pygame.K_RIGHT:
                            self.human_v[0] = speed

                        # Player 2 controls up/down
                        elif event.key == pygame.K_w:
                            self.partner_v[1] = -speed

                        elif event.key == pygame.K_s:
                            self.partner_v[1] = speed

                    # PLAYBACK MODE
                    elif self.mode == PLAYBACK:

                        # Human controls horizontal only
                        if event.key == pygame.K_LEFT:
                            self.human_v[0] = -speed

                        elif event.key == pygame.K_RIGHT:
                            self.human_v[0] = speed

                    # AI MODE
                    elif self.mode == AI:

                        # AI controls vertical
                        if self.ai_control_axis == AI_CONTROLS_VERTICAL:

                            # Human controls horizontal
                            if event.key == pygame.K_LEFT:
                                self.human_v[0] = -speed

                            elif event.key == pygame.K_RIGHT:
                                self.human_v[0] = speed

                        else:

                            # Human controls vertical
                            if event.key == pygame.K_w:
                                self.human_v[1] = -speed

                            elif event.key == pygame.K_s:
                                self.human_v[1] = speed

            # Playback partner movement
            if self.mode == PLAYBACK:
                self.partner_v = self.get_playback_input()

            # AI partner movement
            elif self.mode == AI:
                if self.human_v != [0, 0]:
                    self.partner_v = self.get_ai_input()
                else: 
                    self.partner_v = [0, 0]
                    
            # 2. Joint Action: SUM THE VELOCITIES
            self.cursor_pos[0] += self.human_v[0] + self.partner_v[0]
            self.cursor_pos[1] += self.human_v[1] + self.partner_v[1]

            # Clamp cursor to screen bounds
            self.cursor_pos[0] = max(CURSOR_RADIUS, min(SCREEN_WIDTH - CURSOR_RADIUS, self.cursor_pos[0]))
            self.cursor_pos[1] = max(CURSOR_RADIUS, min(SCREEN_HEIGHT - CURSOR_RADIUS, self.cursor_pos[1]))

            # 3. Check Target Collision
            dist = ((self.cursor_pos[0] - self.target_pos[0]) ** 2 + (self.cursor_pos[1] - self.target_pos[1]) ** 2) ** 0.5

            if dist < (CURSOR_RADIUS + TARGET_RADIUS):

                self.target_hit = True

                print(f"Target Hit! Iteration {self.iteration} complete.")

            # 4. Draw everything
            pygame.draw.circle(self.screen, RED, self.target_pos, TARGET_RADIUS)
            pygame.draw.circle(self.screen, BLUE, (int(self.cursor_pos[0]), int(self.cursor_pos[1])), CURSOR_RADIUS)

            # Draw mode indicator and iteration
            mode_text = f"Mode: {self.mode.upper()} | Iteration: {self.iteration}/{NUM_ITERATIONS}"

            self.draw_text(mode_text, font, BLACK, self.screen, 10, 10)

            # Draw controls info
            if self.mode == INDIVIDUAL:

                self.draw_text("Controls: Arrow Keys", font, BLACK, self.screen, 10, 35)

            elif self.mode == COOPERATIVE:

                self.draw_text("P1: LEFT/RIGHT arrows | P2: W/S keys", font, BLACK, self.screen, 10, 35)

            elif self.mode == PLAYBACK:

                self.draw_text("Playback partner active", font, BLACK, self.screen, 10, 35)

            elif self.mode == AI:
                ai_axis = "UP/DOWN" if self.ai_control_axis == AI_CONTROLS_VERTICAL else "LEFT/RIGHT"
                player_axis = "LEFT/RIGHT" if self.ai_control_axis == AI_CONTROLS_VERTICAL else "UP/DOWN"
                if self.iteration == 1:
                    self.draw_text(f"AI block 1: AI controls {ai_axis} | You control {player_axis}", font, BLACK, self.screen, 10, 35)
                elif self.iteration == (NUM_ITERATIONS // 2) + 1:
                    self.draw_text(f"AXIS CHANGE: AI now controls {ai_axis} | You now control {player_axis}", font, BLACK, self.screen, 10, 35)
                else:
                    self.draw_text(f"AI controls: {ai_axis} | You control: {player_axis}", font, BLACK, self.screen, 10, 35)

            
            
            self.draw_text("Press ESC to quit", font, BLACK, self.screen, 10, 60)

            # 5. Log and Update
            self.log_frame()

            pygame.display.flip()

            self.clock.tick(FPS)

        if self.target_hit:
            self.save_data()


def show_menu():
        pygame.init()
        screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
        pygame.display.set_caption("CogSci Joint Action Task - Mode Selection")
        clock = pygame.time.Clock()
        font = pygame.font.Font(None, 72)
        small_font = pygame.font.Font(None, 42)
        selected = 0
        modes = [(INDIVIDUAL, "1. Individual Mode"), (COOPERATIVE, "2. Cooperative Mode"), (PLAYBACK, "3. Playback Mode"), (AI, "4. AI Mode")]
        selecting_mode = True
        while selecting_mode:
            screen.fill(WHITE)
            title = font.render("Select Game Mode", True, BLACK)
            screen.blit(title, (SCREEN_WIDTH // 2 - title.get_width() // 2, 100))
            start_y = 250
            for i, (mode_key, mode_text) in enumerate(modes):
                color = BLUE if i == selected else BLACK
                prefix = ">>> " if i == selected else "    "
                text = small_font.render(prefix + mode_text, True, color)
                screen.blit(text, (SCREEN_WIDTH // 2 - text.get_width() // 2, start_y + i * 100))
            info_text = small_font.render(f"Each mode will run {NUM_ITERATIONS} times", True, GRAY)
            screen.blit(info_text, (SCREEN_WIDTH // 2 - info_text.get_width() // 2, 680))
            instructions = small_font.render("UP/DOWN = select | ENTER = confirm | ESC = exit", True, GRAY)
            screen.blit(instructions, (SCREEN_WIDTH // 2 - instructions.get_width() // 2, 740))
            pygame.display.flip()
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    pygame.quit()
                    return None
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_UP:
                        selected = (selected - 1) % len(modes)
                    elif event.key == pygame.K_DOWN:
                        selected = (selected + 1) % len(modes)
                    elif event.key == pygame.K_RETURN:
                        selecting_mode = False
                    elif event.key == pygame.K_ESCAPE:
                        pygame.quit()
                        return None
            clock.tick(FPS)
        pygame.quit()
        return modes[selected][0]

while True:
        mode = show_menu()
        if mode is None:
            print("Exiting game.")
            break
        for iteration in range(1, NUM_ITERATIONS + 1):
            ai_axis = None
            if mode == AI:
                if iteration <= NUM_ITERATIONS / 2:
                    ai_axis = AI_CONTROLS_VERTICAL
                else:
                    ai_axis = AI_CONTROLS_HORIZONTAL
            game = ExperimentGame(mode=mode, iteration=iteration, ai_axis=ai_axis)
            game.run()
            if not game.running:
                break

pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html
Target Hit! Iteration 1 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 2 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 3 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 4 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 5 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 6 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 7 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 8 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 9 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 10 complete.
Data saved to experiment_data.csv
Target Hit! Iteration 11 complete.
Data saved to experiment_data.csv
